# Choosing metrics

A benchmark reports numbers; a *useful* benchmark reports numbers that say which
option was better. Most quality metrics in imaging do the first and not the
second, and treating one kind as the other is how a comparison ends up ranking
on noise.

`sprotmetrics` keeps that distinction in the catalogue, where code can read it,
rather than in a spreadsheet where it has to be remembered.

In [1]:
import sprotmetrics as spm

catalogue = spm.list_metrics()
len(catalogue)

51

In [2]:
catalogue[["label", "task", "direction", "strength"]].head(12)

,label,task,direction,strength
metric_id,,,,
hpa_localisation_agreement,Agreement with Human Protein Atlas subcellular...,marker characterisation,higher is better,strong
n_cells,Number of segmented objects,segmentation,descriptive,strong
n_markers,Panel size,dataset statistics,descriptive,strong
qc_pass_rate,Fraction of cells passing QC,QC,descriptive,strong
median_area,Object area,segmentation,descriptive,strong
mean_area,Object area,segmentation,descriptive,strong
median_total_intensity,Per-cell summed intensity,aggregation,descriptive,strong
mean_total_intensity,Per-cell summed intensity,aggregation,descriptive,strong
median_marker_coverage,Per-marker detection rate,image correction,higher is better,weak


## Direction

`direction` records whether a larger value is better, and it has four values
rather than two, because "we do not know" and "it depends" are real answers:

| value | meaning |
| --- | --- |
| `higher is better` / `lower is better` | rankable |
| `descriptive` | real information, no preferred value |
| `ambiguous` | could reflect biology or noise, so the direction is unsafe |

In [3]:
catalogue["direction"].value_counts()

direction
higher is better    20
lower is better     19
descriptive         11
ambiguous            1
Name: count, dtype: int64

`median_area` is the archetypal descriptive metric. An area of 76 against 255
tells you the segmentations differ, not which one you want — and the answer
depends on the tissue.

In [4]:
for metric_id in ["median_area", "median_marker_cv", "spillover_ratio"]:
    info = spm.get_metric(metric_id)
    print(f"{metric_id:26s} {info.direction.value:18s} maximize={info.maximize}")

median_area                descriptive        maximize=None
median_marker_cv           ambiguous          maximize=None
spillover_ratio            lower is better    maximize=False


`maximize` is `None` wherever ranking would be unsound. That is not decoration:
it is what stops a descriptive metric reaching a scoring stage.

## Strength

Some metrics are directional and still cannot carry a decision alone, usually
because something other than quality moves them. These are marked `weak`.

In [5]:
weak = catalogue[catalogue.strength == "weak"]
weak[["label", "direction"]]

,label,direction
metric_id,,
median_marker_coverage,Per-marker detection rate,higher is better
min_marker_coverage,Per-marker detection rate,higher is better
median_marker_zero_fraction,Per-marker fraction at or below zero,lower is better
mean_marker_zero_fraction,Per-marker fraction at or below zero,lower is better
median_marker_dynamic_range,Per-marker dynamic range,higher is better
dapi_positive_ratio,Fraction of objects containing nuclear signal,higher is better
replicate_correlation,Agreement between repeated nuclear stains,higher is better
nuclear_signal_containment,Nuclear signal falling inside any mask,higher is better


In [6]:
print(spm.get_metric("nuclear_signal_containment").notes)

Must be read against spillover_ratio, never alone. Growing every mask raises it, so on exemplar-001 it prefers watershed_expand (0.67) to threshold (0.44) while spillover_ratio says watershed over-expands (1.21 against 1.09). The pair brackets segmentation; either one by itself is gameable.


Growing every mask raises both `nuclear_signal_containment` and
`dapi_positive_ratio`, so each rewards the over-expansion it is meant to help
detect. They are useful read against `spillover_ratio`, which moves the other
way, and misleading read alone. [Comparing two
segmentations](comparing-segmentations) shows this happening on real numbers.

## Filtering to what can rank

Combining both filters gives the metrics that can actually settle a comparison.

In [7]:
rankable = spm.list_metrics(directional_only=True)
strong = rankable[rankable.strength == "strong"]

print(f"{len(catalogue)} metrics")
print(f"{len(rankable)} directional")
print(f"{len(strong)} directional and strong")

51 metrics
39 directional
31 directional and strong


In [8]:
import pandas as pd

# Metrics can score more than one pipeline stage, so split the task column
# rather than grouping on it, and show every stage including the empty ones.
stages = [t.value for t in spm.Task]
counts = strong["task"].str.split(", ").explode().value_counts().reindex(stages).fillna(0).astype(int)
pd.DataFrame({"strong directional metrics": counts})

,strong directional metrics
task,
image correction,10
segmentation,8
aggregation,8
QC,0
normalization,14
marker characterisation,1
dataset statistics,0
all axes,1


Worth reading as a coverage report rather than a score sheet. If a pipeline
stage has no strong directional metric, no amount of computation will tell you
which option to pick for it — you are measuring the other stages and hoping.

QC is the clear gap here, and the reason is structural rather than an oversight:

In [9]:
info = spm.get_metric("qc_pass_rate")
print(f"direction: {info.direction.value}")
print(info.notes)

direction: descriptive
Has no direction and cannot: a no-op QC method scores 1.0 by construction.


A QC method that discards nothing passes 100% of cells, so a higher pass rate
cannot mean a better filter. Making QC rankable needs an external reference —
cells labelled as artefacts by someone — not another statistic derived from the
same data.

## What the package deliberately does not compute

Metrics that were considered and rejected stay in the catalogue with their
reasoning attached, so the question does not have to be re-litigated.

In [10]:
retired = spm.list_metrics(include_deprecated=True).query("status == 'deprecated'")
retired[["label", "superseded_by"]]

,label,superseded_by
metric_id,,
snr_per_protein,Per-marker signal-to-noise ratio,median_marker_cv
cv_per_protein,Per-marker coefficient of variation,median_marker_cv
dynamic_range_per_protein,Per-marker dynamic range,median_marker_dynamic_range
sparsity_per_protein,Per-marker fraction of exact zeros,median_marker_zero_fraction
gene_pair_cosine_similarity,Similarity between two aggregation methods' ou...,
variance_explained_by_total_counts,Variance explained by single-cell technical co...,mean_variance_explained_by_covariate_counts


In [11]:
print(spm.get_metric("snr_per_protein").notes)

Exactly 1 / cv, so it carries no information beyond median_marker_cv. Reporting both would double-count one quantity in any aggregate score.


In [12]:
# Asking for one is an error that names the replacement.
try:
    spm.compute("snr_per_protein", None)
except KeyError as exc:
    print(exc)

'snr_per_protein is deprecated and has no implementation; see median_marker_cv'


## Exporting for a benchmark

For an [OpenProblems](https://openproblems.bio)-style task, the catalogue emits
the `info.metrics[]` block a component config needs, with the direction already
filled in.

In [13]:
block = spm.to_openproblems_info(family="segmentation_quality")
for entry in block["metrics"][:2]:
    print({k: entry[k] for k in ("name", "min", "max", "maximize")})

{'name': 'dapi_positive_ratio', 'min': 0, 'max': 1, 'maximize': True}
{'name': 'dapi_positive_ratio_qc_pass', 'min': 0, 'max': 1, 'maximize': True}


Metrics without a safe direction are left out, because the schema has no way to
express "no direction" — a `maximize` flag is required, and inventing one would
hand the scoring stage a criterion the metric cannot support. The exporter
refuses rather than guesses:

In [14]:
try:
    spm.get_metric("median_area").to_openproblems()
except ValueError as exc:
    print(exc)

median_area is descriptive; the OpenProblems schema has no way to express that, and emitting maximize=true would let the scoring stage rank methods on it. Exclude it, or give it a direction.


## Next

- [Quickstart](quickstart) — computing metrics on your own data.
- [Comparing two segmentations](comparing-segmentations) — the filters above
  applied to an actual decision.